# Exercise 1.3.1.8 — construct instructed-pairs dataset

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.1 Linear Probes`  
**Notebook:** `1.3.1_Linear_Probes_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.1.8](https://delta-drills.vercel.app/?arena_exercise=1.3.1.8)


# [1.3.1] Linear Probes (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/11_[1.3.1]_Linear_Probes)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-31.png" width="350">

# Introduction

This exercise set is built around **linear probing**, one of the most important tools in mechanistic interpretability for understanding what information language models represent internally.

We'll look at three papers:

- The [Geometry of Truth](https://arxiv.org/abs/2310.06824) paper by Marks & Tegmark, which shows that LLMs develop linear representations of truth that generalize across diverse datasets and are causally implicated in model outputs.
- The [deception probes paper](https://arxiv.org/abs/2502.03407) from Apollo Research, which extends this from factual truth to *strategic deception detection* - showing probes trained on simple contrastive data can generalize to realistic deception scenarios.
- The [high-stakes interactions paper](https://arxiv.org/abs/2506.10805) (NeurIPS 2025), which trains attention probes to detect whether a user's *request* is high-stakes - a different target from model intent - and shows they match full LLM classifiers at a fraction of the compute cost.

### What is probing?

The core idea: extract internal activations from a model, then train a simple classifier on them. If a *linear* probe can accurately classify some property from the activations, that property is **linearly represented** in the model's internal state.

From the Geometry of Truth paper:

> *"We identify a linear representation of truth that generalizes across several structurally and topically diverse datasets... these representations are not merely associated with truth, but are also causally implicated in the model's output."*

The "causally implicated" part matters a lot: it's not just that we can read off truth from model internals, but that the model actually *uses* these representations when computing outputs. We'll verify this in Section 3.

### Why this matters for safety

If we can reliably detect truth, deception, or intent from model internals, there are direct implications for model monitoring. [Neel Nanda argues](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in) probes could even be used *during training*:

> *"There are certain things that may be much easier to specify using the internals of the model. For example: Did it do something for the right reasons? Did it only act this way because it knew it was being trained or watched?"*

But this is genuinely controversial. The worry, as [Bronson Schoen puts it](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in?commentId=CtZnXwZuBgcWsagwn), is that training against probe signals might just teach the model to hide whatever the probe was measuring:

> *"If you train directly against non-obfuscated internals and no longer see bad behavior, the obvious possibility is that now you've just got obfuscated internals."*

For deception probes specifically, the generalization question is especially pointed: we may need to detect sophisticated deception in scenarios we've never seen. As the Apollo paper puts it: *"our monitors will need to exhibit generalization - correctly identifying deceptive text in new types of scenarios."*

### What you'll build

These exercises cover the full pipeline: extracting activations, visualizing with PCA, training probes, validating them causally, and applying them to deception detection. Layer choice, token position, and probe type all matter - part of the point is developing intuition for *why*.

### Models we'll use

Sections 1-3 use `meta-llama/Llama-2-13b-hf` (base model, ~26GB in bfloat16). The Geometry of Truth paper has specific configurations for this model (`probe_layer=14`, `intervene_layer=8`), so our results should closely match theirs. Section 4 switches to `meta-llama/Meta-Llama-3.1-8B-Instruct` (instruct-tuned, ~16GB), needed for the deception detection instructed-pairs methodology.

Both fit comfortably on a single A100. If you have a multi-GPU setup, the 70B variants are worth trying as a bonus; the paper's strongest results are at that scale.

## Content & Learning Objectives

### 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

### 2️⃣ Training & comparing probes

> ##### Learning Objectives
>
> * Implement difference-of-means (MM) and logistic regression (LR) probes
> * Compare probe types: accuracy, direction similarity, and what each captures
> * Understand CCS (Contrastive Consistent Search) as an unsupervised alternative and its limitations

### 3️⃣ Causal interventions

> ##### Learning Objectives
>
> * Understand why classification accuracy alone is insufficient - causal evidence is needed
> * Implement activation patching with probe directions to flip model predictions
> * Compare the causal effects of MM vs. LR probe directions
> * Appreciate that MM probes find more causally implicated directions despite lower classification accuracy

### 4️⃣ Probing for Deception

> ##### Learning Objectives
>
> * Construct instructed-pairs datasets following the deception-detection paper's methodology
> * Train deception probes on instruct-tuned models
> * Evaluate whether deception probes generalize to factual truth/falsehood datasets
> * Understand methodological choices that affect replicability

### 5️⃣ Attention Probes for High-Stakes Detection

> ##### Learning Objectives
>
> * Understand what "high-stakes interactions" means as a probe target, and why it differs from probing model intent
> * Extract full-sequence activations (shape `(n, seq, d_model)`) rather than last-token only
> * Implement an attention probe as a `nn.Module` - a single learned query that computes a weighted sum over token positions before classification
> * Compare attention pooling against last-token and mean-pool baselines using AUROC
> * Inspect learned attention weights to understand which parts of a prompt are most diagnostic

## Reading Material

The core papers we'll be replicating are "The Geometry of Truth" and "Detecting Strategic Deception Using Linear Probes" - you should at least skim both before starting, so you understand the basics. The other references give you more context on the probing literature and the open questions around it.

- [The Geometry of Truth: Emergent Linear Structure in Large Language Model Representations of True/False Datasets](https://arxiv.org/abs/2310.06824) by Marks & Tegmark (COLM 2024). Shows that LLMs develop linear representations of truth that generalise across diverse datasets and are causally implicated in model outputs. Read at least the abstract, and sections 1, 2 & 4 (intro, datasets & visualization). You can skip 3, because we won't be doing much of this kind of patching in these exercises.
- [Detecting Strategic Deception Using Linear Probes](https://arxiv.org/abs/2502.03407) by Goldowsky-Dill et al. (Apollo Research, 2025). Extends truth probing to *strategic deception detection*, showing that probes trained on simple contrastive data can generalise to realistic deception scenarios. Section 4 of this exercise set replicates their methodology. Read the abstract and sections 1 & 3 (introduction and methodology).
- [Detecting High-Stakes Interactions with Activation Probes](https://arxiv.org/abs/2506.10805) by McKenzie et al. (NeurIPS 2025). Trains attention probes to detect whether a user's request is high-stakes, matching full LLM classifiers at much lower cost. Section 5 of this exercise set replicates this. Read the abstract and section 2 (methodology), or alternatively just the [EleutherAI post](https://blog.eleuther.ai/attention-probes/).
- [Discovering Latent Knowledge in Language Models Without Supervision](https://arxiv.org/abs/2212.03827) by Burns et al. (ICLR 2023). Introduces Contrastive Consistent Search (CCS), an unsupervised method for finding truth directions without labelled data. We discuss CCS limitations in section 2 but don't implement it in full. Optional reading.

## Setup code

Before running this, you'll need to clone the Geometry of Truth as well as Deception Detection repos into the `exercises` directory:

```bash
cd chapter1_transformer_interp/exercises

git clone https://github.com/saprmarks/geometry-of-truth.git
git clone https://github.com/ApolloResearch/deception-detection.git
```

`Llama-2-13b-hf` is a gated model, so you'll need a HuggingFace access token (as well as requesting access [here](https://huggingface.co/meta-llama/Llama-2-13b-hf)). When you've got access and made a HuggingFace token, create a `.env` file in your `chapter1_transformer_interp/exercises` directory with:

```
HF_TOKEN=hf_your_token_here
```

then the code below will use this token for authentication.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = t.device("cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part31_linear_probes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part31_linear_probes.tests as tests
import part31_linear_probes.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# Set up paths to the cloned repos
# Adjust these if your repos are in a different location
GOT_ROOT = exercises_dir / "geometry-of-truth"  # geometry-of-truth repo
DD_ROOT = exercises_dir / "deception-detection"  # deception-detection repo

assert GOT_ROOT.exists(), f"Please clone geometry-of-truth repo to {GOT_ROOT}"
assert DD_ROOT.exists(), f"Please clone deception-detection repo to {DD_ROOT}"

GOT_DATASETS = GOT_ROOT / "datasets"
DD_DATA = DD_ROOT / "data"

### Loading the model

We start with LLaMA-2-13B, a base (not instruction-tuned) model. The Geometry of Truth paper uses this model with `probe_layer=14` and `intervene_layer=8` - we'll use these exact values.

In [ ]:
load_dotenv(dotenv_path=str(exercises_dir / ".env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your chapter1_transformer_interp/exercises/.env file"

In [ ]:
MODEL_NAME = "meta-llama/Llama-2-13b-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
# Layer choices from the geometry-of-truth repo config for llama-2-13b. The paper
# found truth representations are concentrated in early-to-mid layers, and identified
# these specific layers via patching experiments (Section 3, "group (b)").
PROBE_LAYER = 14
INTERVENE_LAYER = 8

print(f"Model: {MODEL_NAME}")
print(f"Layers: {NUM_LAYERS}, Hidden dim: {D_MODEL}")
print(f"Probe layer: {PROBE_LAYER}, Intervene layer: {INTERVENE_LAYER}")

### Loading the datasets

The Geometry of Truth paper uses several carefully curated datasets of simple true/false statements. Each dataset has a `statement` column and a `label` column (1=true, 0=false).

From the paper:
> *"We find that the truth-related structure in LLM representations is much cleaner for our curated datasets than for our unstructured ones."*

Let's load three of these curated datasets and examine them:

In [ ]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(GOT_DATASETS / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))

# 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

## Extracting activations

Our first task is to extract hidden state activations from the model. For the Geometry of Truth approach, we extract the **last-token** activation at each specified layer. For declarative statements like "The city of Paris is in France.", the model's representation of whether the statement is true or false is concentrated at the final token position.

Note - the Geometry of Truth paper probes specifically at the **end-of-sentence punctuation** token (the period / full stop). The datasets are designed so that every statement ends with a period, meaning the last token is always the period. This is important because the model's truth representation builds up over the sentence and is concentrated at the final punctuation mark.

A few technical details to keep in mind. We use `output_hidden_states=True` in the forward pass to get all layer activations. `outputs.hidden_states` has length `num_layers + 1`: index 0 is the embedding output, and index `i` for `i >= 1` is the output of layer `i-1`. We also need to handle **padding** correctly, since statements have different lengths. We pad them but must extract the activation at the last *real* (non-padding) token, not the last position.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.1.8"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part31_linear_probes.solutions import extract_activations, get_pca_components, layer_sweep_accuracy, MMProbe, LRProbe, compute_generalization_matrix, few_shot_evaluate, intervention_experiment


### Exercise - construct instructed-pairs dataset

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> Getting the dataset construction right is critical for replicability.
> ```

Create paired prompts using the `you_are_fact_sys` variant from the deception-detection repo. The honest version uses system="You are an honest AI assistant." and the dishonest version uses system="You are a deceptive AI assistant." Both use user="Please tell me a fact." with assistant=[fact_start].

Here `fact_start` is the statement **minus the last 5 words** (the last 5 words are excluded to capture the model's "intent" before it commits to the full statement, following the paper).

For each pair, extract activations from the specified layers and **average across detected tokens** (the tokens corresponding to `fact_start`) to get one vector per dialogue.

<details>
<summary>Hint - splitting statements</summary>

Split by spaces: `words = statement.split(" ")`, then `start = " ".join(words[:-5])` and `end = " ".join(words[-5:])`. The detect region is the tokens of `start` only.
</details>

<details>
<summary>Hint - using ChatActivations</summary>

Use `ChatActivations.from_messages(messages, tokenizer)` to build a detection mask over the assistant content, then call `chat_acts.extract_activations(model, layers)` to get the averaged activation vector for each layer.
</details>

In [ ]:
def construct_instructed_pairs(
    facts: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
) -> tuple[dict[int, Float[Tensor, "n d"]], dict[int, Float[Tensor, "n d"]]]:
    """
    Construct instructed-pairs dataset and extract averaged activations.

    For each fact, creates an honest and dishonest version using the you_are_fact_sys format.
    Uses ChatActivations to robustly identify the assistant-content tokens, then extracts and
    averages activations over those tokens.

    Args:
        facts: List of factual statements.
        model: Instruct-tuned language model.
        tokenizer: Corresponding tokenizer.
        layers: Layers to extract activations from.

    Returns:
        Tuple of (honest_acts, dishonest_acts), each a dict mapping layer -> [n_facts, d_model] tensor.
    """
    honest_acts = {layer: [] for layer in layers}
    dishonest_acts = {layer: [] for layer in layers}

    # "you_are_fact_sys" variant from the repo - system prompt sets the persona, user asks
    # for a fact. The repo tested several prompt variants; this is the default.
    honest_sys = "You are an honest AI assistant."
    dishonest_sys = "You are a deceptive AI assistant."
    user_msg = "Please tell me a fact."

    for fact in tqdm(facts, desc="Extracting instructed-pairs"):
        # Exclude the last 5 words of each fact (following the paper). The idea is that
        # we don't want the probe to see the actual truth/falsehood of the completed fact;
        # we only want to capture the model's "intention to deceive" which the system prompt
        # primes it for. The paper says: "We gather activations on this fact excluding the
        # last 5 words, so that the fact could be completed in either a truthful or
        # deceptive manner." (Note: the paper says "tokens" but the implementation excludes words.)
        words = fact.split(" ")
        fact_start = " ".join(words[:-5]) if len(words) > 5 else fact

        for sys_prompt, acts_dict in [(honest_sys, honest_acts), (dishonest_sys, dishonest_acts)]:
            messages = [
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": fact_start},
            ]

            # YOUR CODE HERE - use ChatActivations.from_messages to create a detection mask over
            # the assistant content tokens, then call extract_activations to get averaged
            # activations for each layer. Append each layer's activation to acts_dict[layer].
            raise NotImplementedError()

    honest_out = {layer: t.stack(acts_list) for layer, acts_list in honest_acts.items()}
    dishonest_out = {layer: t.stack(acts_list) for layer, acts_list in dishonest_acts.items()}
    return honest_out, dishonest_out


all_facts = true_facts["statement"].tolist()

# Split into train/test (shuffle to avoid ordering bias in the CSV)
t.manual_seed(42)
n_train = int(0.8 * len(all_facts))
perm = t.randperm(len(all_facts))
train_facts = [all_facts[i] for i in perm[:n_train]]
test_facts = [all_facts[i] for i in perm[n_train:]]

# Extract activations (use a single representative layer for speed, plus full detect range)
# For initial testing, use just the middle layer
mid_layer = INSTRUCT_NUM_LAYERS // 2

train_honest, train_dishonest = construct_instructed_pairs(
    train_facts, instruct_model, instruct_tokenizer, [mid_layer]
)
test_honest, test_dishonest = construct_instructed_pairs(
    test_facts, instruct_model, instruct_tokenizer, [mid_layer]
)

# Show first few pairs
pairs_df = pd.DataFrame(
    {
        "Fact": train_facts[:3],
        "Honest norm": [f"{train_honest[mid_layer][i].norm():.1f}" for i in range(3)],
        "Dishonest norm": [f"{train_dishonest[mid_layer][i].norm():.1f}" for i in range(3)],
    }
)
display(pairs_df)

<details><summary>Solution</summary>

```python
def construct_instructed_pairs(
    facts: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
) -> tuple[dict[int, Float[Tensor, "n d"]], dict[int, Float[Tensor, "n d"]]]:
    """
    Construct instructed-pairs dataset and extract averaged activations.

    For each fact, creates an honest and dishonest version using the you_are_fact_sys format.
    Uses ChatActivations to robustly identify the assistant-content tokens, then extracts and
    averages activations over those tokens.

    Args:
        facts: List of factual statements.
        model: Instruct-tuned language model.
        tokenizer: Corresponding tokenizer.
        layers: Layers to extract activations from.

    Returns:
        Tuple of (honest_acts, dishonest_acts), each a dict mapping layer -> [n_facts, d_model] tensor.
    """
    honest_acts = {layer: [] for layer in layers}
    dishonest_acts = {layer: [] for layer in layers}

    # "you_are_fact_sys" variant from the repo - system prompt sets the persona, user asks
    # for a fact. The repo tested several prompt variants; this is the default.
    honest_sys = "You are an honest AI assistant."
    dishonest_sys = "You are a deceptive AI assistant."
    user_msg = "Please tell me a fact."

    for fact in tqdm(facts, desc="Extracting instructed-pairs"):
        # Exclude the last 5 words of each fact (following the paper). The idea is that
        # we don't want the probe to see the actual truth/falsehood of the completed fact;
        # we only want to capture the model's "intention to deceive" which the system prompt
        # primes it for. The paper says: "We gather activations on this fact excluding the
        # last 5 words, so that the fact could be completed in either a truthful or
        # deceptive manner." (Note: the paper says "tokens" but the implementation excludes words.)
        words = fact.split(" ")
        fact_start = " ".join(words[:-5]) if len(words) > 5 else fact

        for sys_prompt, acts_dict in [(honest_sys, honest_acts), (dishonest_sys, dishonest_acts)]:
            messages = [
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": fact_start},
            ]

            chat_acts = ChatActivations.from_messages(messages, tokenizer)
            layer_acts = chat_acts.extract_activations(model, layers, average=True)
            for layer in layers:
                acts_dict[layer].append(layer_acts[layer])

    honest_out = {layer: t.stack(acts_list) for layer, acts_list in honest_acts.items()}
    dishonest_out = {layer: t.stack(acts_list) for layer, acts_list in dishonest_acts.items()}
    return honest_out, dishonest_out
```
</details>

### A note on layer selection

The deception probes paper's pre-trained probe (`detector.pt`) uses layer 22 of 80 in Llama-3.3-70B-Instruct, roughly the 28th percentile of model depth. This is notably earlier than the middle of the network. The repo's default configuration uses the **middle 50% of layers** (layers 20-59 for an 80-layer model) for multi-layer aggregation, but the best single-layer probe was found at layer 22.

We use `mid_layer = INSTRUCT_NUM_LAYERS // 2` (layer 16 for the 32-layer 8B model) as a starting point, but note that naively scaling the layer index proportionally across model sizes (e.g. 22/80 to 9/32) is not necessarily correct. Smaller models may need a certain absolute number of layers before forming the higher-order representations that deception probes rely on; the relevant features may not emerge until a minimum depth regardless of total model size. The paper does not include 8B results, so there is no published guidance on the optimal layer for this model size.

The Geometry of Truth paper also picks specific layers per model size without proportional scaling: `probe_layer=14` for the 40-layer LLaMA-2-13B and `probe_layer=27` for the 80-layer LLaMA-2-70B, both in the lower third of the network.

### Bonus exercise - multi-layer aggregation

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
>
> This is a bonus exercise for students who want to explore layer sensitivity.
> ```

The deception-detection repo trains a separate probe at each layer and **averages their scores** to get a final per-dialogue score (see `by_layer.py` in the repo). This multi-layer aggregation is more robust than relying on a single layer, since the optimal layer varies across datasets and model sizes.

Try the following:
1. Re-run `construct_instructed_pairs` with `layers=INSTRUCT_DETECT_LAYERS` (the middle 50% of layers) instead of just `[mid_layer]`.
2. Train a separate `LogisticRegression` probe at each layer.
3. For each test example, compute the mean score across all layer probes.
4. Compare the AUROC of this multi-layer aggregated score against the single-layer probe.

You should find that multi-layer aggregation is at least as good as the best single layer, and sometimes better, since it provides insurance against picking a suboptimal layer. The paper's `by_layer.py` script shows that performance can vary dramatically between adjacent layers, reinforcing that a sweep or aggregation strategy is important.

We now train the deception probes. We reuse the `MMProbe` and `LRProbe` classes from Section 2. For the MM probe the direction is simply mean(dishonest) - mean(honest). For the LR probe we use `C=0.001` (matching `reg_coeff=1000` in the deception-detection repo's defaults), which is stronger regularization than the truth probe from Section 2.

In [ ]:
# Build training data: dishonest=1 (positive class), honest=0 (negative class)
train_all_acts = t.cat([train_dishonest[mid_layer], train_honest[mid_layer]], dim=0)
train_all_labels = t.cat([t.ones(len(train_dishonest[mid_layer])), t.zeros(len(train_honest[mid_layer]))])

mm_probe = MMProbe.from_data(train_all_acts, train_all_labels)
lr_probe = LRProbe.from_data(train_all_acts, train_all_labels, C=0.001)

# Evaluate MM on train/test
train_all = t.cat([train_dishonest[mid_layer], train_honest[mid_layer]])
train_y = t.cat([t.ones(len(train_dishonest[mid_layer])), t.zeros(len(train_honest[mid_layer]))])
test_all = t.cat([test_dishonest[mid_layer], test_honest[mid_layer]])
test_y = t.cat([t.ones(len(test_dishonest[mid_layer])), t.zeros(len(test_honest[mid_layer]))])

mms_train_acc = (mm_probe.pred(train_all) == train_all_labels).float().mean().item()
mms_test_acc = (mm_probe.pred(test_all) == test_y).float().mean().item()

# Evaluate LR on train/test
lr_train_acc = (lr_probe.pred(train_all) == train_all_labels).float().mean().item()
lr_test_acc = (lr_probe.pred(test_all) == test_y).float().mean().item()

assert mms_train_acc > 0.6, f"MM train accuracy too low: {mms_train_acc:.3f}"
assert lr_train_acc > 0.6, f"LR train accuracy too low: {lr_train_acc:.3f}"

probe_results = pd.DataFrame(
    {
        "Probe": ["MM", "LR"],
        "Train Acc": [f"{mms_train_acc:.3f}", f"{lr_train_acc:.3f}"],
        "Test Acc": [f"{mms_test_acc:.3f}", f"{lr_test_acc:.3f}"],
    }
)
print("Deception probe accuracy:")
display(probe_results)

# PCA scatter of honest vs dishonest
all_acts = t.cat([train_honest[mid_layer], train_dishonest[mid_layer]])
pcs = get_pca_components(all_acts, k=2)
centered = all_acts - all_acts.mean(dim=0)
projected = (centered @ pcs).numpy()
n_honest = len(train_honest[mid_layer])
colors = ["green"] * n_honest + ["red"] * (len(all_acts) - n_honest)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=projected[:n_honest, 0],
        y=projected[:n_honest, 1],
        mode="markers",
        marker=dict(color="green", size=4, opacity=0.5),
        name="Honest",
    )
)
fig.add_trace(
    go.Scatter(
        x=projected[n_honest:, 0],
        y=projected[n_honest:, 1],
        mode="markers",
        marker=dict(color="red", size=4, opacity=0.5),
        name="Dishonest",
    )
)
fig.update_layout(
    title="PCA of Honest vs Dishonest Activations", xaxis_title="PC1", yaxis_title="PC2", height=450, width=600
)
fig.show()

# Histogram of LR scores on test set
lr_scores_test = lr_probe(test_all).detach().numpy()
n_test_dis = len(test_dishonest[mid_layer])
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=lr_scores_test[n_test_dis:], name="Honest", marker_color="green", opacity=0.6, nbinsx=20)
)
fig.add_trace(
    go.Histogram(x=lr_scores_test[:n_test_dis], name="Dishonest", marker_color="red", opacity=0.6, nbinsx=20)
)
fig.update_layout(
    title="LR Probe Score Distribution (Test Set)",
    xaxis_title="P(Deceptive)",
    yaxis_title="Count",
    barmode="overlay",
    height=400,
    width=600,
)
fig.show()

<details>
<summary>Question - Why might the MM and LR probes show similar accuracy here, even though their directions can differ substantially?</summary>

Both probes are trained on the same contrastive data (honest vs. dishonest system prompts), and for this binary classification the decision boundary is well-determined. The Geometry of Truth paper found that classification accuracy often doesn't distinguish probe types: *"LR, MM, and CCS probes all having very similar sp_en_trans classification accuracies."*

The more interesting differences show up in *causal effects* (which we tested in Section 3 for truth probes) and out-of-distribution behavior. The MM direction tends to be more robust because it captures the geometric center of the honest/dishonest clusters rather than optimizing a decision boundary that might exploit spurious features.
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
